# v31 教師なし分析 — 近傍幾何転写の可視化 / datum双峰のNCC観察 / typewellファミリー
- **A**: `nbr_geo_v31.py` の結果(nbr_geo_preds.npz)をグラフ化(per-well RMSE ECDF, 近傍距離 vs 精度, 事例井戸)。
  ※ A は先に `python v31/nbr_geo_v31.py` を実行しておく(B/C は単独で動く)。
- **B**: 学習なしの**定数オフセットNCC走査**(dip直線+δ で typewell と相関)→ 双峰構造を直接観察
  (souldrive の「第二極小・コインフリップ」を自データで再現+可視化)。
- **C**: typewell の**同一ファイル(ハッシュ)ファミリー**をクラスタ → ファミリー内で datum 誤差が共有されるか診断。
- 入力は `refs_cache.npz`(train_aligner_v31 cell2 で構築済)+CSV軽読み。PF不使用。


In [ ]:
# ===== 共通ロード(refs_cache + XY重心 + typewellハッシュ) =====
import glob, json, math, time, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
warnings.filterwarnings("ignore")
for f in ["Meiryo", "Yu Gothic", "MS Gothic"]:
    try:
        matplotlib.rcParams["font.family"] = f; break
    except Exception: pass
matplotlib.rcParams["axes.unicode_minus"] = False

PROJECT  = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
DATA_DIR = PROJECT / "rogii-wellbore-geology-prediction" / "train"
ART = PROJECT / "v31" / "artifacts_v31"

_z = np.load(ART / "refs_cache.npz", allow_pickle=True)
WD_ = {w: _z[w].item() for w in _z.files}
WELLS = sorted(WD_)
print("refs cache:", len(WELLS), "wells")

meta_fp = ART / "unsup_meta.json"
if meta_fp.exists():
    META = json.loads(meta_fp.read_text())
else:
    META = {}
    t0 = time.time()
    for w in WELLS:
        hw = pd.read_csv(DATA_DIR / f"{w}__horizontal_well.csv", usecols=["X", "Y"])
        h = hashlib.md5((DATA_DIR / f"{w}__typewell.csv").read_bytes()).hexdigest()
        META[w] = dict(cx=float(hw["X"].mean()), cy=float(hw["Y"].mean()), twh=h)
    meta_fp.write_text(json.dumps(META))
    print(f"meta built ({time.time()-t0:.0f}s)")

DIP_TAU = 500.0   # dip外挿の減衰スケール(線形外挿は暴走する: nbr_geoで実証)

def dip_line(w):
    """ヒールdipの減衰付き外挿(drift = ir*TAU*(1-exp(-dMD/TAU)))。"""
    d = WD_[w]; a = d["pre"]
    if a is None or len(a) < 8:
        return np.full(len(d["q_md"]), d["lk"])
    t = a[-30:, 3]; z = a[-30:, 1]; m = a[-30:, 0]
    dt = np.diff(t + z); dm = np.diff(m); ok = dm > 0
    ir = float(np.median(dt[ok] / dm[ok])) if ok.sum() >= 3 else 0.0
    geo0 = float(a[-1, 3] + a[-1, 1])
    drift = ir * DIP_TAU * (1.0 - np.exp(-np.maximum(d["q_md"] - float(a[-1, 0]), 0.0) / DIP_TAU))
    return geo0 + drift - d["q_z"]

def rmse(e):
    e = np.asarray(e); e = e[np.isfinite(e)]
    return float(np.sqrt(np.mean(e ** 2))) if e.size else np.nan

DIP = {w: dip_line(w) for w in WELLS}
per_dip = {w: rmse(DIP[w] - WD_[w]["tvt_true"]) for w in WELLS}
print(f"dip直線のみ overall={rmse(np.concatenate([DIP[w]-WD_[w]['tvt_true'] for w in WELLS])):.3f}")


In [ ]:
# ===== A: nbr_geo(近傍幾何転写)結果の可視化 =====
fpA = ART / "nbr_geo_preds.npz"
if not fpA.exists():
    print("nbr_geo_preds.npz が無い → 先に python v31/nbr_geo_v31.py を実行")
else:
    zz = np.load(fpA, allow_pickle=True)
    NG = {w: zz[f"p_{w}"] for w in WELLS if f"p_{w}" in zz.files}
    per_ng = {w: rmse(NG[w] - WD_[w]["tvt_true"]) for w in NG}
    # 最近傍井戸距離(重心)
    xy = np.array([[META[w]["cx"], META[w]["cy"]] for w in WELLS])
    nnd = {}
    for i, w in enumerate(WELLS):
        d = np.hypot(xy[:, 0] - xy[i, 0], xy[:, 1] - xy[i, 1]); d[i] = np.inf
        nnd[w] = float(d.min())
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
    # (1) per-well RMSE の ECDF
    for vals, lab in [(list(per_ng.values()), "nbr_geo"), (list(per_dip.values()), "dip直線")]:
        v = np.sort([x for x in vals if np.isfinite(x)])
        ax[0].plot(v, np.arange(1, len(v) + 1) / len(v), label=f"{lab} (ov={rmse(np.concatenate([ (NG if lab=='nbr_geo' else DIP)[w]-WD_[w]['tvt_true'] for w in (NG if lab=='nbr_geo' else DIP)])):.2f})")
    ax[0].axvline(10, ls="--", c="gray"); ax[0].set_xlim(0, 40)
    ax[0].set_xlabel("per-well RMSE"); ax[0].set_ylabel("ECDF"); ax[0].legend(); ax[0].set_title("per-well RMSE ECDF")
    # (2) 最近傍距離 vs RMSE
    xs = [nnd[w] for w in NG]; ys = [per_ng[w] for w in NG]
    ax[1].scatter(xs, ys, s=8, alpha=0.5)
    ax[1].set_xscale("log"); ax[1].set_ylim(0, 40)
    ax[1].set_xlabel("最近傍井戸距離 [ft] (重心)"); ax[1].set_ylabel("nbr_geo RMSE")
    ax[1].axhline(10, ls="--", c="gray"); ax[1].axvline(1500, ls=":", c="red")
    ax[1].set_title("近傍距離 vs 精度(1500ft=転写半径)")
    # (3) 改善量ヒスト(dip直線比)
    dd = [per_dip[w] - per_ng[w] for w in NG if np.isfinite(per_dip[w]) and np.isfinite(per_ng[w])]
    ax[2].hist(dd, bins=60, range=(-15, 15)); ax[2].axvline(0, c="k")
    ax[2].set_xlabel("RMSE改善 (dip直線 − nbr_geo)"); ax[2].set_title(f"改善井戸 {sum(1 for x in dd if x>0)}/{len(dd)}")
    plt.tight_layout(); plt.show()
    # (4) 事例: nbr_geo が効いた/効かなかった井戸(近傍あり側から)
    cov = [w for w in NG if nnd[w] < 1500]
    good = sorted(cov, key=lambda w: per_ng[w] - per_dip[w])[:2]
    bad = sorted(cov, key=lambda w: per_dip[w] - per_ng[w])[:2]
    fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
    for k, w in enumerate(good + bad):
        d = WD_[w]
        ax[k].plot(d["q_md"], d["tvt_true"], "k", lw=1.5, label="true")
        ax[k].plot(d["q_md"], NG[w], "r", lw=1, label=f"nbr_geo {per_ng[w]:.1f}")
        ax[k].plot(d["q_md"], DIP[w], "b--", lw=1, label=f"dip {per_dip[w]:.1f}")
        ax[k].set_title(("good: " if k < 2 else "bad: ") + w[:8]); ax[k].legend(fontsize=7)
        ax[k].invert_yaxis(); ax[k].set_xlabel("MD")
    plt.tight_layout(); plt.show()


In [ ]:
# ===== B: 定数オフセットNCC走査 — datum双峰の直接観察(学習なし・PFなし) =====
DELTAS = np.arange(-40, 40.001, 0.5)

def offset_scan(w):
    """dip直線+δ の軌跡で typewell GR と相関 → corr(δ)。"""
    d = WD_[w]
    gt = d["grid_tvt"]; twc = d["tw"]
    gr = d["q_gr"].astype(np.float64)
    base = DIP[w]  # dip直線TVT
    ok0 = np.isfinite(gr) & np.isfinite(base)
    if ok0.sum() < 50: return None
    g = gr[ok0]; b = base[ok0]
    cors = np.full(len(DELTAS), np.nan)
    for i, dl in enumerate(DELTAS):
        e = np.interp(b + dl, gt, np.nan_to_num(twc, nan=np.nan), left=np.nan, right=np.nan)
        m = np.isfinite(e)
        if m.sum() < 50: continue
        gg = g[m]; ee = e[m]
        sg = gg.std(); se = ee.std()
        if sg < 1e-6 or se < 1e-6: continue
        cors[i] = float(np.corrcoef(gg, ee)[0, 1])
    return cors

SCAN = {}; MODES = {}
t0 = time.time()
for w in WELLS:
    c = offset_scan(w)
    if c is None or not np.isfinite(c).any(): continue
    SCAN[w] = c
    cc = np.nan_to_num(c, nan=-1)
    pk, _ = find_peaks(cc, distance=20)             # 峰間 >10ft
    pk = pk[np.argsort(-cc[pk])][:2]
    d = WD_[w]
    dtrue = float(np.nanmedian(d["tvt_true"] - DIP[w]))
    MODES[w] = dict(mu=[float(DELTAS[p]) for p in pk], cor=[float(cc[p]) for p in pk], dtrue=dtrue)
print(f"scan {len(SCAN)} wells ({time.time()-t0:.0f}s)")

bi = {w: m for w, m in MODES.items() if len(m["mu"]) == 2 and abs(m["mu"][0] - m["mu"][1]) >= 8}
uni = {w: m for w, m in MODES.items() if w not in bi}
print(f"双峰(第2峰あり, 峰間>=8ft): {len(bi)} / 単峰扱い: {len(uni)}")

# 図1: 事例 corr カーブ(双峰3 + 単峰3)
ex_bi = sorted(bi, key=lambda w: -abs(bi[w]["mu"][0] - bi[w]["mu"][1]))[:3]
ex_un = sorted(uni, key=lambda w: -max(uni[w]["cor"] or [0]))[:3]
fig, ax = plt.subplots(1, 6, figsize=(19, 3), sharey=True)
for k, w in enumerate(ex_bi + ex_un):
    ax[k].plot(DELTAS, SCAN[w]); m = MODES[w]
    for mu in m["mu"]: ax[k].axvline(mu, c="r", ls=":")
    ax[k].axvline(m["dtrue"], c="g", lw=2, alpha=0.6)
    ax[k].set_title(("双峰 " if k < 3 else "単峰 ") + w[:8], fontsize=9)
    ax[k].set_xlabel("datum shift δ [ft]")
ax[0].set_ylabel("corr(GR, typewell)")
plt.suptitle("緑=真のdatum / 赤点線=NCC峰"); plt.tight_layout(); plt.show()

# 図2: 双峰井戸の統計(souldriveのコインフリップ検証)
gaps = [abs(m["mu"][0] - m["mu"][1]) for m in bi.values()]
correct1 = [abs(m["mu"][0] - m["dtrue"]) < abs(m["mu"][1] - m["dtrue"]) for m in bi.values()]
margin = [m["cor"][0] - m["cor"][1] for m in bi.values()]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(gaps, bins=30); ax[0].set_xlabel("峰間距離 [ft]"); ax[0].set_title(f"双峰の峰間分布 (n={len(bi)})")
ax[1].scatter(margin, [1 if c else 0 for c in correct1], s=10, alpha=0.4)
ax[1].set_xlabel("corr margin (峰1−峰2)"); ax[1].set_yticks([0, 1]); ax[1].set_yticklabels(["峰2が正", "峰1が正"])
ax[1].set_title(f"最大峰の正解率 = {np.mean(correct1):.2f}(0.5=コインフリップ)")
err1 = [abs(m["mu"][0] - m["dtrue"]) for m in MODES.values()]
errB = [min(abs(mu - m["dtrue"]) for mu in m["mu"]) for m in MODES.values()]
ax[2].hist([np.clip(err1, 0, 30), np.clip(errB, 0, 30)], bins=30, label=["最大峰", "oracle峰"])
ax[2].legend(); ax[2].set_xlabel("|峰 − 真datum| [ft]"); ax[2].set_title("峰選択の誤差(全井戸)")
plt.tight_layout(); plt.show()
np.savez_compressed(ART / "datum_scan.npz",
                    **{f"c_{w}": SCAN[w] for w in SCAN},
                    meta=json.dumps({w: {k: v for k, v in m.items()} for w, m in MODES.items()}))
print("datum_scan.npz 保存(NCC走査カーブ+峰情報=GBM特徴/事後の素材)")


In [ ]:
# ===== C: typewellファミリー(同一ファイルhash)と family prior の価値診断 =====
from collections import Counter, defaultdict
fam = defaultdict(list)
for w in WELLS: fam[META[w]["twh"]].append(w)
sizes = sorted((len(v) for v in fam.values()), reverse=True)
multi = {h: ws for h, ws in fam.items() if len(ws) >= 2}
print(f"typewellファミリー: {len(fam)} 家族 / 2本以上 {len(multi)} 家族"
      f"(井戸数 {sum(len(v) for v in multi.values())}/{len(WELLS)})")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].hist(sizes, bins=range(1, max(sizes) + 2)); ax[0].set_xlabel("ファミリーサイズ")
ax[0].set_title("同一typewellファミリーの分布"); ax[0].set_yscale("log")
# XYマップ(上位ファミリー色分け)
top = sorted(multi, key=lambda h: -len(multi[h]))[:12]
xy = {w: (META[w]["cx"], META[w]["cy"]) for w in WELLS}
ax[1].scatter([xy[w][0] for w in WELLS], [xy[w][1] for w in WELLS], s=4, c="lightgray")
for j, h in enumerate(top):
    ws = multi[h]
    ax[1].scatter([xy[w][0] for w in ws], [xy[w][1] for w in ws], s=14, label=f"F{j}({len(ws)})")
ax[1].set_title("XY配置(上位12ファミリー)"); ax[1].legend(fontsize=6, ncol=2)
# family内で datum誤差(dtrue=真のdatum shift)が共有されるか
if "MODES" in globals():
    ing = []; outg = []
    dts = {w: MODES[w]["dtrue"] for w in MODES}
    for h, ws in multi.items():
        ws2 = [w for w in ws if w in dts]
        if len(ws2) >= 2:
            v = [dts[w] for w in ws2]
            ing.append(float(np.std(v)))
    glob_std = float(np.std(list(dts.values())))
    ax[2].hist(ing, bins=30); ax[2].axvline(glob_std, c="r", label=f"全体std={glob_std:.1f}")
    ax[2].set_xlabel("ファミリー内 datum shift の std [ft]")
    ax[2].set_title(f"family内std中央値={np.median(ing):.1f}(全体より小さいほど family prior が有効)")
    ax[2].legend()
else:
    ax[2].text(0.1, 0.5, "先に B(datum走査)を実行", fontsize=12)
plt.tight_layout(); plt.show()
json.dump({h: ws for h, ws in multi.items()},
          open(ART / "tw_families.json", "w"))
print("tw_families.json 保存(family ID は GBM のカテゴリ特徴候補)")
